# CUDA: Vector Addition
**Objective:** Implement vector addition on GPU using CUDA C.

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Write CUDA kernel
cuda_code = """
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(float *A, float *B, float *C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int n = 100000;
    size_t bytes = n * sizeof(float);
    
    // Host arrays
    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C = (float*)malloc(bytes);
    
    // Initialize
    for (int i = 0; i < n; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }
    
    // Device arrays
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);
    
    // Copy to device
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);
    
    // Execute kernel
    int blockSize = 256;
    int gridSize = (n + blockSize - 1) / blockSize;
    vectorAdd<<<gridSize, blockSize>>>(d_A, d_B, d_C, n);
    
    // Copy result back
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);
    
    // Verify
    printf("First 10 results: ");
    for (int i = 0; i < 10; i++) {
        printf("%.1f ", h_C[i]);
    }
    printf("\\n");
    
    // Cleanup
    free(h_A); free(h_B); free(h_C);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    
    return 0;
}
"""

with open('vector_add.cu', 'w') as f:
    f.write(cuda_code)
print('CUDA code saved')

In [ ]:
# Compile CUDA code
!nvcc -o vector_add vector_add.cu

In [ ]:
# Run
!./vector_add